In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime

# 폴더 생성
os.makedirs('database', exist_ok=True)

DB_PATH = 'database/ESGdata.db'

def setup_database():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # 테이블 생성 (이전 설계 반영)
    cursor.executescript("""
    CREATE TABLE IF NOT EXISTS master_sites (site_id TEXT PRIMARY KEY, site_name TEXT);
    CREATE TABLE IF NOT EXISTS standard_usage (
        id INTEGER PRIMARY KEY AUTOINCREMENT, site_id TEXT, reporting_date TEXT, 
        metric TEXT, value REAL, v_status INTEGER DEFAULT 0
    );
    CREATE TABLE IF NOT EXISTS activity_data (
        activity_id INTEGER PRIMARY KEY AUTOINCREMENT, site_id TEXT, reporting_date TEXT, 
        production_qty REAL, unit TEXT DEFAULT 'Ton'
    );
    """)
    
    # 마스터 데이터 삽입
    cursor.execute("INSERT OR REPLACE INTO master_sites VALUES ('Site A', '대전공장'), ('Site B', '울산공장')")
    
    # CSV 데이터를 DB에 적재
    for site_id in ['Site A', 'Site B']:
        df = pd.read_csv(f'data/site_{site_id[-1].lower()}_test_raw.csv')
        for _, row in df.iterrows():
            date_str = row['date'][:7] # YYYY-MM
            
            # 에너지 데이터 (Electricity)
            cursor.execute("INSERT INTO standard_usage (site_id, reporting_date, metric, value) VALUES (?, ?, ?, ?)",
                           (site_id, date_str, 'electricity_mwh', row['electricity_mwh']))
            # 에너지 데이터 (LNG)
            cursor.execute("INSERT INTO standard_usage (site_id, reporting_date, metric, value) VALUES (?, ?, ?, ?)",
                           (site_id, date_str, 'lng_nm3', row['lng_nm3']))
            # 활동 데이터 (Production)
            cursor.execute("INSERT INTO activity_data (site_id, reporting_date, production_qty) VALUES (?, ?, ?)",
                           (site_id, date_str, row['production_ton']))
            
    conn.commit()
    conn.close()
    print(f"✅ DB 적재 완료: {DB_PATH}")

if __name__ == "__main__":
    setup_database()

✅ DB 적재 완료: database/ESGdata.db
